- Plot clean URLs as the background, then overlay the adversarial URLs --> found out whether the 3-character edits move a URL from the phishing side toward the legit side.

In [10]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, tldextract
import json, torch, torch.nn as nn
from transformers import BertConfig, BertModel, BertTokenizerFast
import sys
sys.path.insert(0, "..")
from src.load_data import load_splits
from sklearn.decomposition import PCA
from sklearn.neighbors import KNeighborsClassifier
import torch
from pathlib import Path
import re

SEED = 42
N_PER_SET = 2000          # points sampled per set (prevent overplotting)
MAX_GROUP_FRAC = 1.00     
N_ARROWS = 150            # orig -> adv arrows drawn per adversarial set
TOP_GROUPS = 5            # color for top 5 domain groups; the rest -> "other"
GROUP_BY = "registered_domain"   
PROJECTION = "pca"       # or "umap"
OUT_DIR = "../embedding_plots"

ARCHS = ["CNN", "BiLSTM", "URLBERT"]
LABEL_NAMES = {0: "phishing", 1: "legitimate"}

import os; os.makedirs(OUT_DIR, exist_ok=True)
_tld = tldextract.TLDExtract(suffix_list_urls=())  

# Load data
- adversarial - keep only rows that were edited and successful fooled model (flipped)

In [11]:
import sys; sys.path.append("../src")
def canon(url):
    url = re.sub(r"^[A-Za-z][A-Za-z0-9+.\-]*://", "", str(url).strip())
    return re.sub(r"/+$", "", re.sub(r"^www\d*\.", "", url, flags=re.IGNORECASE))

_b = lambda s: s.astype(str).eq("True") 

def read_orig(path):
    df = pd.read_csv(path, keep_default_na=False).rename(columns={"URL": "url"})
    df["url"] = df["url"].map(canon)
    return df[["url", "label"]]                   

ORIG = {"train_orig": read_orig("../data/train.csv"), "test_orig": read_orig("../data/test.csv")}

def read_adv(path, split_urls, only_flipped=True):
    df = pd.read_csv(path, keep_default_na=False).rename(columns={"orig_URL": "orig_url", "URL": "adv_url"})
    n0 = len(df); df["n_edits"] = df["n_edits"].astype(int)
    dup = _b(df["duplicate"]) if "duplicate" in df else (df.n_edits > 0) & (df.adv_url.isin(split_urls) | df.adv_url.duplicated())
    keep = (df.n_edits > 0) & ~dup
    if only_flipped: keep &= _b(df["flipped"])
    df = df[keep].assign(url=lambda d: d.adv_url, label=0)
    print(f"{Path(path).name}: kept {len(df)}/{n0}")
    return df[["url", "label", "orig_url"]].reset_index(drop=True)

A = Path("../data/adversarial"); tr, te = set(ORIG["train_orig"].url), set(ORIG["test_orig"].url)
ADV = {
    "CNN": dict(train_adv=read_adv(A/"cnn_train_adversarial.csv", tr),
                test_adv1=read_adv(A/"cnn_test_adversarial.csv", te),
                test_adv2=read_adv(A/"cnn_m2_test_adversarial.csv", te)),
    "URLBERT": dict(train_adv=read_adv("../data/urlbert_adv_train.csv", tr),
                    test_adv1=read_adv("../data/urlbert_adv_test.csv", te), test_adv2=read_adv("../data/urlbert_model2_adv_test.csv", te)),
    "BiLSTM": dict(train_adv=read_adv("../data/bilstm_adv_train.csv", tr), test_adv1=read_adv("../data/bilstm_adv_test.csv", te), test_adv2=read_adv("../data/bilstm_m2_adv_test.csv", te)), 
}
DATA_BY_ARCH = {a: {**ORIG, **s} for a, s in ADV.items()}
def get_data(arch, name): return DATA_BY_ARCH[arch].get(name)

cnn_train_adversarial.csv: kept 884/15983
cnn_test_adversarial.csv: kept 333/7939
cnn_m2_test_adversarial.csv: kept 54/7847
urlbert_adv_train.csv: kept 1481/15476
urlbert_adv_test.csv: kept 3047/7729
urlbert_model2_adv_test.csv: kept 344/7951
bilstm_adv_train.csv: kept 2526/15899
bilstm_adv_test.csv: kept 739/7876
bilstm_m2_adv_test.csv: kept 64/7876


# Encoder
-  maps a list of URL strings -> np.ndarray (N, D) of PENULTIMATE-layer features (the vector that feeds the final classification layer)
- Dimensions also differ across CNN, BiLSTM and URLBERT - normalised the shift by each model’s class-centroid gap. Compare the ratios across architectures, not raw distances.

## note: penultimate layer has a built-in bias  
- trained to separate the classes, so S1 will look cleanly separated. 
- The informative parts are where the adversarial points land and the structure within each class. 
- Features the model discarded won’t show up --> absence of domain clustering there doesn’t prove the model ignores domain.

In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def torch_encoder(model, tokenize, device, bs=256):
    # Returns the INPUT to the final nn.Linear (the penultimate vector)
    head = [m for m in model.modules() if isinstance(m, torch.nn.Linear)][-1]
    store = {}
    head.register_forward_hook(lambda m, inp, out: store.__setitem__("x", inp[0].detach().float().cpu()))
    model = model.to(device).eval()

    @torch.no_grad()
    def enc(urls):
        out = []
        for i in range(0, len(urls), bs):
            batch = tokenize(urls[i:i + bs])
            if isinstance(batch, dict):
                model(**{k: v.to(device) for k, v in batch.items()})
            else:
                model(batch.to(device))
            out.append(store["x"].numpy())
        return np.vstack(out)

    enc.model, enc.head = model, head  
    return enc

def build(ckpt, loader, tokenize):
    # ckpt: path to the saved weights; loader: function that builds the model and loads them
    # (load_cnn / load_urlbert); tokenize: char_tokenize / bert_tokenize.
    if not Path(ckpt).exists():
        print(f"skip: {ckpt} not found"); return None
    return torch_encoder(loader(ckpt), tokenize, DEVICE)

# ---------- CNN ----------
vocab = json.load(open("../data/vocab.json")); cfg = json.load(open("../data/config.json"))
char_to_idx = {c: i for i, c in enumerate(vocab["index_to_string"])}
PAD_IDX, UNK_IDX, CHAR_MAX_LEN, VOCAB_SIZE = vocab["pad_idx"], vocab["unk_idx"], cfg["max_len"], len(char_to_idx)

class CharacterCNN(nn.Module):
    def __init__(self, vocab_size, embedding_dim=32):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=PAD_IDX)
        self.conv1 = nn.Conv1d(embedding_dim, 128, kernel_size=5); self.conv2 = nn.Conv1d(128, 128, kernel_size=5)
        self.relu = nn.ReLU(); self.pool = nn.AdaptiveMaxPool1d(1); self.fc = nn.Linear(128, 1)
    def forward(self, x):
        x = self.embedding(x).transpose(1, 2)
        x = self.relu(self.conv1(x)); x = self.relu(self.conv2(x))
        return self.fc(self.pool(x).squeeze(2))

def char_tokenize(urls):
    arr = np.full((len(urls), CHAR_MAX_LEN), PAD_IDX, dtype=np.int64)
    for r, u in enumerate(map(canon, urls)):
        row = [char_to_idx.get(c, UNK_IDX) for c in u[:CHAR_MAX_LEN]]; arr[r, :len(row)] = row
    return torch.from_numpy(arr)

def load_cnn(path):
    m = CharacterCNN(VOCAB_SIZE, 32); m.load_state_dict(torch.load(path, map_location=DEVICE, weights_only=True)); return m

# ---------- URLBERT ----------
URLBERT_DIR = "../models/urlbert"
tok = BertTokenizerFast(vocab_file=f"{URLBERT_DIR}/bert_tokenizer/vocab.txt", do_lower_case=True)
_lens = [len(x) for x in tok(load_splits()["train"].URL.tolist())["input_ids"]]
BERT_MAX_LEN = min(int(np.percentile(_lens, 99)), 128)  # same rule as training

class URLBERTClassifier(nn.Module):
    def __init__(self, encoder):
        super().__init__(); self.encoder = encoder; self.dropout = nn.Dropout(0.1); self.fc = nn.Linear(768, 2)
    def forward(self, input_ids, attention_mask):
        h = self.encoder(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state[:, 0]
        return self.fc(self.dropout(h))

def load_urlbert(path):
    c = BertConfig.from_json_file(f"{URLBERT_DIR}/bert_config/config.json"); c.vocab_size = 5000
    m = URLBERTClassifier(BertModel(c, add_pooling_layer=False))
    m.load_state_dict(torch.load(path, map_location="cpu", weights_only=True)); return m

def bert_tokenize(urls):
    e = tok(urls, truncation=True, max_length=BERT_MAX_LEN, padding=True, return_tensors="pt")
    return {"input_ids": e["input_ids"], "attention_mask": e["attention_mask"]} 

# ---------- BiLSTM ----------
import generate_bilstm_adversarial as attack_impl
def load_bilstm(path):
    return attack_impl.load_model(Path(path), VOCAB_SIZE, PAD_IDX, DEVICE)

ENCODERS = {
    "CNN": {"m1": build("../models/cnn_baseline.pt",        load_cnn, char_tokenize),
            "m2": build("../models/cnn_adv_trained_s42.pt", load_cnn, char_tokenize)},
    "BiLSTM": {"m1": build("../models/bilstm_model1.pt", load_bilstm, char_tokenize)), "m2": None}, #
    "URLBERT": {"m1": build("../checkpoints/urlbert_model1.pt", load_urlbert, bert_tokenize),                                           
                "m2": build("../checkpoints/urlbert_model2.pt", load_urlbert, bert_tokenize)},
}

skip: ../models/cnn_adv_trained_s42.pt not found
skip: ../checkpoints/urlbert_model2.pt not found


In [ ]:
def domain_group(url): #color by subdomains
    e = _tld(url)
    return e.registered_domain if GROUP_BY == "registered_domain" else e.suffix

_cache = {}
def embed(arch, mkey, urls):
    c = _cache.setdefault((arch, mkey), {})
    todo = [u for u in dict.fromkeys(urls) if u not in c]
    if todo:
        for u, e in zip(todo, ENCODERS[arch][mkey](todo)): c[u] = e
    return np.stack([c[u] for u in urls])

def sample_set(df, adv, n=N_PER_SET):
    #sample N points from each set to prevent overplotting
    df = df.copy()
    df["group"] = df["orig_url" if adv else "url"].map(domain_group)
    if not adv: 
        cap = max(1, int(MAX_GROUP_FRAC * n)) #change MAX_GROUP_FRAC if one subdomain is dominating plots
        df = df.sample(frac=1, random_state=SEED)                       # shuffle, then keep at most `cap` rows per group
        df = df[df.groupby("group").cumcount() < cap]
    return df.sample(min(len(df), n), random_state=SEED).reset_index(drop=True)

SCENARIOS = {
    "S1  MODEL 1 | train_orig":                   dict(model="m1", orig="train_orig", advs=[]),
    "S2  MODEL 2 | train_orig + train_adv":        dict(model="m2", orig="train_orig", advs=["train_adv"]),
    "S3  MODEL 1 | test_orig + test_adv(M1)":      dict(model="m1", orig="test_orig",  advs=["test_adv1"]),
    "S4  MODEL 2 | test_orig + test_adv(M1,M2)":   dict(model="m2", orig="test_orig",  advs=["test_adv1", "test_adv2"]),
}
ADV_STYLE = {"train_adv": ("tab:orange", "train adv"), "test_adv1": ("tab:orange", "adv (vs M1)"), "test_adv2": ("tab:purple", "adv (vs M2)")}

def make_projector(ref_emb): #fit clean points on projector, adversarial points are added onto plot without reshaping plot
    pca = PCA(n_components=min(50, ref_emb.shape[1]), random_state=SEED).fit(ref_emb)
    if PROJECTION == "umap":
        import umap
        um = umap.UMAP(n_neighbors=30, min_dist=0.2, random_state=SEED).fit(pca.transform(ref_emb))
        return (lambda X: um.transform(pca.transform(X))), um.embedding_
    p2 = PCA(n_components=2, random_state=SEED).fit(ref_emb)
    return p2.transform, p2.transform(ref_emb)

def run_scenario(arch, sc):
    mkey = sc["model"]
    if ENCODERS[arch][mkey] is None: return None
    ref = sample_set(get_data(arch, sc["orig"]), adv=False)
    ref_emb = embed(arch, mkey, ref["url"].tolist())
    project, ref_xy = make_projector(ref_emb) # fit on ORIGINAL points only; adversarial points are projected in
    res = dict(ref=ref, ref_emb=ref_emb, ref_xy=ref_xy, advs={})
    for name in sc["advs"]:
        df = get_data(arch, name)
        if df is None: continue
        a = sample_set(df, adv=True)
        a_emb = embed(arch, mkey, a["url"].tolist())
        o_emb = embed(arch, mkey, a["orig_url"].tolist())
        res["advs"][name] = dict(df=a, emb=a_emb, xy=project(a_emb), orig_emb=o_emb, orig_xy=project(o_emb))
    return res

def draw(ax, res, by):
    ref, xy = res["ref"], res["ref_xy"]
    if by == "class": # legit = blue, phishing = red
        for lab, col in [(1, "tab:blue"), (0, "tab:red")]:
            m = (ref["label"] == lab).values
            ax.scatter(*xy[m].T, s=4, c=col, alpha=.35, label=f"orig {LABEL_NAMES[lab]}")
        for name, a in res["advs"].items():
            col, lbl = ADV_STYLE[name]
            ax.scatter(*a["xy"].T, s=10, c=col, marker="x", alpha=.7, label=lbl) # adversarial = x, arrows from original --> adversarial (show how point shifted)
            for i in range(min(N_ARROWS, len(a["xy"]))):
                ax.annotate("", xy=a["xy"][i], xytext=a["orig_xy"][i], arrowprops=dict(arrowstyle="->", color=col, alpha=.25, lw=.5))
    else:
        top = ref["group"].value_counts().index[:TOP_GROUPS] #color by subdomain
        cmap = plt.get_cmap("tab10")
        for i, g in enumerate(list(top) + ["other"]):
            m = (ref["group"].isin(top) == False).values if g == "other" else (ref["group"] == g).values
            ax.scatter(*xy[m].T, s=4, color=cmap(i) if g != "other" else "lightgrey", alpha=.5, label=g)
        for name, a in res["advs"].items():
            ax.scatter(*a["xy"].T, s=10, c="k", marker="x", alpha=.5, label=f"{ADV_STYLE[name][1]}")
    ax.set_xticks([]); ax.set_yticks([])

def shift_metrics(res, k=15): 
    # look at 15 nearest original points of adversarial url in the full vector space 
    # reports fraction of legit url before and after the edit & how far the vector moved.
    ref, E = res["ref"], res["ref_emb"]
    knn = KNeighborsClassifier(n_neighbors=k).fit(E, ref["label"].values)
    gap = np.linalg.norm(E[ref["label"] == 0].mean(0) - E[ref["label"] == 1].mean(0))
    rows = []
    for name, a in res["advs"].items():
        p_orig_legit = knn.predict_proba(a["orig_emb"])[:, list(knn.classes_).index(1)].mean() if 1 in knn.classes_ else np.nan
        p_adv_legit  = knn.predict_proba(a["emb"])[:, list(knn.classes_).index(1)].mean() if 1 in knn.classes_ else np.nan
        disp = np.linalg.norm(a["emb"] - a["orig_emb"], axis=1)
        rows.append(dict(adv_set=name, n=len(a["emb"]),
                         knn_legit_frac_orig=p_orig_legit,    # neighbourhood of the clean phishing URL
                         knn_legit_frac_adv=p_adv_legit,      # neighbourhood after perturbation
                         median_shift_over_class_gap=np.median(disp) / gap))
    return pd.DataFrame(rows)

In [ ]:
metrics = []
for arch in ARCHS:
    fig, axes = plt.subplots(2, len(SCENARIOS), figsize=(5.2 * len(SCENARIOS), 10.5))
    for j, (title, sc) in enumerate(SCENARIOS.items()):
        res = run_scenario(arch, sc)
        if res is None:
            for i in range(2): axes[i, j].text(.5, .5, "model not available", ha="center"); axes[i, j].axis("off")
            axes[0, j].set_title(title, fontsize=9); continue
        draw(axes[0, j], res, "class"); draw(axes[1, j], res, "group")
        axes[0, j].set_title(title, fontsize=9)
        axes[0, j].legend(fontsize=6, markerscale=2, loc="best"); axes[1, j].legend(fontsize=6, markerscale=2, loc="best")
        m = shift_metrics(res)
        if len(m): m.insert(0, "scenario", title.split("|")[0].strip()); m.insert(0, "arch", arch); metrics.append(m)
    fig.suptitle(f"{arch}: penultimate-layer embeddings ({PROJECTION.upper()}, fit on original points only)", y=1.0)
    fig.tight_layout(); fig.savefig(f"{OUT_DIR}/{arch}_embeddings.png", dpi=160, bbox_inches="tight"); plt.show()

metrics_df = pd.concat(metrics) if metrics else pd.DataFrame()
metrics_df.to_csv(f"{OUT_DIR}/shift_metrics.csv", index=False)
metrics_df

# Tensorflow

In [ ]:
def export_projector(arch, scenario_title, outdir=f"{OUT_DIR}/projector"):
    sc = SCENARIOS[scenario_title]; res = run_scenario(arch, sc)
    E = [res["ref_emb"]]; meta = [pd.DataFrame(dict(kind="orig", label=res["ref"]["label"].map(LABEL_NAMES), group=res["ref"]["group"], url=res["ref"]["url"]))]
    for name, a in res["advs"].items():
        E.append(a["emb"]); meta.append(pd.DataFrame(dict(kind=name, label="phishing(adv)", group=a["df"]["group"], url=a["df"]["url"])))
    os.makedirs(outdir, exist_ok=True)
    tag = f"{arch}_{scenario_title.split()[0]}"
    np.savetxt(f"{outdir}/{tag}_vecs.tsv", np.vstack(E), delimiter="\t")
    pd.concat(meta).to_csv(f"{outdir}/{tag}_meta.tsv", sep="\t", index=False)
    print("wrote", tag)

In [ ]:
export_projector("CNN", "S4  MODEL 2 | test_orig + test_adv(M1,M2)")